# Session 2 — N-body: Five Implementations
**Numerical & Scientific Computing — AAU**

This notebook shows the **minimal implementation** of the N-body force step for each paradigm.
Take what you need — build your own race.

| # | Paradigm | Key idea |
|---|----------|----------|
| 1 | NumPy naive | Python loop over particles |
| 2 | NumPy vectorized | Full NxN broadcast, no Python loop |
| 3 | Numba `prange` | JIT-compiled, parallel outer loop |
| 4 | Dask | Chunked force computation, thread pool |
| 5 | GPU (CuPy + Numba CUDA) | Same broadcast on GPU / explicit kernel |

---

In [7]:
import numpy as np
import time

# ── Simulation parameters ─────────────────────────────────────────────────────
G   = 1.0    # gravitational constant
M   = 1.0    # particle mass
EPS = 0.1    # softening (avoids singularity at r=0)
DT  = 0.005  # timestep

def init_galaxy(N, seed=42):
    """Random disk of N particles with approximate circular velocities."""
    rng = np.random.default_rng(seed)
    r   = 2.0 * np.sqrt(rng.uniform(0.05, 1.0, N))
    phi = rng.uniform(0, 2*np.pi, N)
    pos = np.zeros((N, 3)); vel = np.zeros((N, 3))
    pos[:,0] = r*np.cos(phi); pos[:,1] = r*np.sin(phi)
    pos[:,2] = rng.normal(0, 0.05, N)
    v = np.sqrt(G*M*N / (r+EPS))
    vel[:,0] = -v*np.sin(phi); vel[:,1] = v*np.cos(phi)
    return pos.astype(np.float64), vel.astype(np.float64)

# Reference forces (used for correctness checks)
def forces_ref(pos):
    diff      = pos[np.newaxis,:,:] - pos[:,np.newaxis,:]   # (N,N,3)
    dist2     = np.sum(diff**2, axis=2) + EPS**2
    inv_dist3 = dist2**(-1.5)
    return G*M**2 * np.sum(diff * inv_dist3[:,:,np.newaxis], axis=1)

def check(label, F, F_ref, tol=1e-6):
    err = np.max(np.abs(np.asarray(F) - np.asarray(F_ref)))
    ok  = err < tol
    print(f"  {label}: max|err| = {err:.2e}  {'✓' if ok else f'✗ (tol={tol:.0e})'}")

N_TEST = 50
pos_t, vel_t = init_galaxy(N_TEST)
F_ref = forces_ref(pos_t)
print(f"Setup OK — N_TEST={N_TEST}")

Setup OK — N_TEST=50


---
## 1 — NumPy Naive
One Python `for` loop over particles. Inner `j` computation is vectorized.
Still O(N) interpreter iterations — slowest, but easiest to read.

In [2]:
def forces_naive(pos):
    N = pos.shape[0]
    F = np.zeros_like(pos)
    for i in range(N):
        diff     = pos - pos[i]                      # (N,3) — vector from i to all j
        dist2    = np.sum(diff**2, axis=1) + EPS**2  # (N,)
        inv3     = dist2**(-1.5)
        inv3[i]  = 0.0                               # zero self-force
        F[i]     = G * M**2 * np.dot(inv3, diff)
    return F

def step_naive(pos, vel):
    F = forces_naive(pos)
    vel += F / M * DT
    pos += vel * DT
    return pos, vel

check('naive', forces_naive(pos_t), F_ref)

  naive: max|err| = 1.78e-14  ✓


---
## 2 — NumPy Vectorized
Full NxN broadcast — no Python loop. One `(N, N, 3)` array, all operations in NumPy C.
Memory cost: O(N²). Works well up to N ~ 5000 before RAM becomes an issue.

In [3]:
def forces_vectorized(pos):
    diff      = pos[np.newaxis,:,:] - pos[:,np.newaxis,:]  # (N,N,3)
    dist2     = np.sum(diff**2, axis=2) + EPS**2            # (N,N)
    inv_dist3 = dist2**(-1.5)                                # (N,N)
    # diff[i,i] = 0 → self-force = 0 automatically
    return G * M**2 * np.sum(diff * inv_dist3[:,:,np.newaxis], axis=1)  # (N,3)

def step_vectorized(pos, vel):
    F = forces_vectorized(pos)
    vel += F / M * DT
    pos += vel * DT
    return pos, vel

check('vectorized', forces_vectorized(pos_t), F_ref)

  vectorized: max|err| = 0.00e+00  ✓


---
## 3 — Numba `prange` (multi-core)
`@njit(parallel=True)` + `prange` parallelizes the outer `i` loop across all CPU cores.
Inner `j` loop is pure C — no Python overhead. First call triggers JIT compilation (~2s).

In [4]:
from numba import njit, prange

@njit(parallel=True, fastmath=True)
def forces_prange(pos, G, M, EPS):
    N    = pos.shape[0]
    F    = np.zeros_like(pos)
    Gm2  = G * M * M
    eps2 = EPS * EPS
    for i in prange(N):              # ← parallel over particles
        fx = fy = fz = 0.0
        xi = pos[i,0]; yi = pos[i,1]; zi = pos[i,2]
        for j in range(N):           # ← sequential: all j per thread
            dx = pos[j,0]-xi; dy = pos[j,1]-yi; dz = pos[j,2]-zi
            d3 = (dx*dx + dy*dy + dz*dz + eps2)**(-1.5)
            fx += Gm2*dx*d3; fy += Gm2*dy*d3; fz += Gm2*dz*d3
        F[i,0] = fx; F[i,1] = fy; F[i,2] = fz
    return F

def step_prange(pos, vel):
    F = forces_prange(pos, G, M, EPS)
    vel += F / M * DT
    pos += vel * DT
    return pos, vel

forces_prange(pos_t, G, M, EPS)   # warmup (JIT compile)
check('prange', forces_prange(pos_t, G, M, EPS), F_ref)

  prange: max|err| = 7.11e-15  ✓


---
## 4 — Dask (distributed / thread pool)
Split the N particles into chunks. Each chunk computes forces for its own particles
using the full position array. Chunks run in parallel via `dask.delayed`.

**Key overhead:** task scheduling (~0.5–1ms per task) + barrier at each timestep.
Dask wins when N is large or when you have a real cluster — not for N < 500 on one node.

In [5]:
import dask

def _chunk_forces(pos_chunk, pos_all):
    diff      = pos_all[np.newaxis,:,:] - pos_chunk[:,np.newaxis,:]  # (k,N,3)
    dist2     = np.sum(diff**2, axis=2) + EPS**2
    inv_dist3 = dist2**(-1.5)
    return G*M**2 * np.sum(diff * inv_dist3[:,:,np.newaxis], axis=1)

def forces_dask(pos, n_chunks=4):
    chunks    = np.array_split(pos, n_chunks, axis=0)
    delayed_f = [dask.delayed(_chunk_forces)(c, pos) for c in chunks]
    return np.vstack(dask.compute(*delayed_f, scheduler='threads'))

def step_dask(pos, vel, n_chunks=4):
    F = forces_dask(pos, n_chunks)
    vel += F / M * DT
    pos += vel * DT
    return pos, vel

check('dask', forces_dask(pos_t, n_chunks=4), F_ref)

  dask: max|err| = 0.00e+00  ✓


---
## 5a — CuPy (GPU vectorized)
Identical idiom to `forces_vectorized` — just swap `np` → `cp`.
CuPy dispatches the broadcast as a fused CUDA kernel automatically.
Data must live on the GPU (`cp.array`). H→D copy happens once before the time loop.

In [6]:
import cupy as cp

def forces_cupy(pos_cp):
    diff      = pos_cp[cp.newaxis,:,:] - pos_cp[:,cp.newaxis,:]  # on GPU
    dist2     = cp.sum(diff**2, axis=2) + EPS**2
    inv_dist3 = dist2**(-1.5)
    return G*M**2 * cp.sum(diff * inv_dist3[:,:,cp.newaxis], axis=1)

def step_cupy(pos_cp, vel_cp):
    F       = forces_cupy(pos_cp)
    vel_cp += F / M * DT
    pos_cp += vel_cp * DT
    return pos_cp, vel_cp

pos_cp = cp.array(pos_t)
F_cp   = forces_cupy(pos_cp)
check('cupy', cp.asnumpy(F_cp), F_ref)

ModuleNotFoundError: No module named 'cupy'

---
## 5b — Numba CUDA (explicit kernel)
One CUDA thread per particle. Each thread loops over all N particles to accumulate its force.
Uses `float32` for GPU speed. First call triggers JIT → CUDA C compilation.

In [8]:
from numba import cuda, float32
import warnings
from numba.core.errors import NumbaPerformanceWarning

@cuda.jit
def _nbody_kernel(pos, forces, N, Gm2, eps2):
    i = cuda.grid(1)
    if i >= N:
        return
    fx = fy = fz = float32(0.0)
    xi = pos[i,0]; yi = pos[i,1]; zi = pos[i,2]
    for j in range(N):
        dx = pos[j,0]-xi; dy = pos[j,1]-yi; dz = pos[j,2]-zi
        d3 = (dx*dx + dy*dy + dz*dz + eps2)**float32(-1.5)
        fx += Gm2*dx*d3; fy += Gm2*dy*d3; fz += Gm2*dz*d3
    forces[i,0] = fx; forces[i,1] = fy; forces[i,2] = fz

BLOCK = 256

def step_cuda(pos_d, vel_d, frc_d, N):
    _nbody_kernel[(N+BLOCK-1)//BLOCK, BLOCK](
        pos_d, frc_d, N, float32(G*M*M), float32(EPS*EPS))
    cuda.synchronize()
    # in-place update using CuPy views (no host round-trip)
    p = cp.asarray(pos_d); v = cp.asarray(vel_d); f = cp.asarray(frc_d)
    v += f / float32(M) * float32(DT)
    p += v * float32(DT)

# Warmup + verify (suppress grid-size warning for tiny N_TEST)
p32   = pos_t.astype(np.float32)
pos_d = cuda.to_device(p32)
frc_d = cuda.device_array_like(p32)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', NumbaPerformanceWarning)
    _nbody_kernel[(N_TEST+BLOCK-1)//BLOCK, BLOCK](
        pos_d, frc_d, N_TEST, float32(G*M*M), float32(EPS*EPS))
cuda.synchronize()
# float32 vs float64: expect ~1e-5 error from reduced precision
check('cuda (f32)', frc_d.copy_to_host(), F_ref.astype(np.float32), tol=1e-3)

CUDA unavailable: skipping 5b kernel warmup/verification in this environment.


---
## Your Turn — Build the Race

You now have all 5 implementations. Your task:

1. **Pick your N values** — suggested: `[100, 200, 500, 1000, 2000]`
2. **Time each paradigm** for T=10 steps using `time.perf_counter()`
3. **Plot** time-per-step vs N on a log-log scale
4. **Answer:** at which N does each paradigm win?

Template:
```python
N_VALUES = [100, 200, 500, 1000, 2000]
T = 10
results = {}   # paradigm → list of ms/step

for N in N_VALUES:
    pos, vel = init_galaxy(N)
    t0 = time.perf_counter()
    for _ in range(T):
        pos, vel = step_vectorized(pos, vel)   # ← swap for other paradigms
    ms_per_step = (time.perf_counter() - t0) / T * 1000
    print(f"N={N}  {ms_per_step:.3f} ms/step")
```

**Export results to CSV** (so we can compare the whole class):
```python
import pandas as pd, socket
df = pd.DataFrame(results)   # your results dict
df.to_csv(f'race_{socket.gethostname()}.csv', index=False)
```